In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import minimize
import statsmodels.api as sm

plt.rcParams["figure.dpi"] = 120

In [ ]:
pip install rpy2

In [ ]:
from rpy2.robjects.packages import importr
import rpy2.robjects as robjects
from rpy2.robjects import numpy2ri

utils = importr("utils")
utils.chooseCRANmirror(ind=1) # this was missing
utils.install_packages('rugarch')
rugarch = importr('rugarch')

In [ ]:
rv = pd.read_csv('SPY.csv')
rv = rv[["Date", "Volatility", "Type"]]
rv.rename(columns={"Volatility": "RV_daily"},inplace=True)
rv = rv[rv['Type'] == 'QMLE-Trade']
rv.drop(columns=['Type'], inplace=True)
rv = rv.set_index("Date")
rv.index = pd.to_datetime(rv.index)
rv.index.name = "date"

rv.head()

In [5]:
# PERMNO = 84398 for SPY
ret = pd.read_csv("returns (crsp).csv")
ret.columns = ret.columns.str.strip()

date_col = "date" if "date" in ret.columns else "Date"
ret_col  = "RET" if "RET" in ret.columns else "RETX"  # prefer RET, fallback to RETX

ret = ret[[date_col, "PERMNO", ret_col]].copy()
ret = ret[ret["PERMNO"] == 84398].drop(columns=["PERMNO"])

ret[date_col] = pd.to_datetime(ret[date_col], format="%d/%m/%Y", errors="raise")
ret = ret.sort_values(date_col).set_index(date_col)
ret.index.name = "date"

ret["r"] = pd.to_numeric(ret[ret_col], errors="coerce")
ret.drop(columns=[ret_col], inplace=True)
ret.dropna(subset=["r"], inplace=True)

ret = ret.loc[:"2024-12-29"]

ret.head()


,r
date,
1996-01-02,0.010673
1996-01-03,0.002766
1996-01-04,-0.009529
1996-01-05,-0.002025
1996-01-08,0.003805


In [8]:
df = rv.join(ret, how="inner").dropna()
df.head()


,RV_daily,r
date,,
1996-01-02,0.140261,0.010673
1996-01-03,0.082399,0.002766
1996-01-04,0.211454,-0.009529
1996-01-05,0.022647,-0.002025
1996-01-09,0.228727,-0.017185
